# Reproducing the results of *Minimal Surfaces, Knots, and Neural Networks*

**Companion notebook to** [arXiv:2605.26234](https://arxiv.org/pdf/2605.26234)·
*Tancredi Schettini Gherardini & Marco Usula, 2026.*

This notebook is intended for a reader who wants to reproduce
the empirical results, figures, and tables of the paper end-to-end. It uses
the trained PINN checkpoints shipped in `trained_models/paper/` to

1. evaluate the tension-field residual of each near-minimal disc,
2. locate self-intersections via the self-proximity map and refine them by Newton's method,
3. read off the signed self-intersection count and compare with the prediction
   of Fine's conjecture (the coefficient of $z^{2g}\,a^{2(g+d)}$ in the HOMFLY
   polynomial $P(K)$).

The notebook is organised to mirror the paper:

| Notebook section | Paper section |
|---|---|
| §1 Introduction; Fine's conjecture | §1 |
| §2 Minimal surfaces in hyperbolic 4-space | §2 |
| §3.1 PINNs · §3.2 The model | §3.1 — §3.2 |
| §3.3 The training regime | §3.3 |
| §3.4 Double-point analysis | §3.4 |
| §4 Results — the main reproduction loop | §4 |
| §5 Future developments | §5 |

Mathematical foundations are recapped briefly; for full statements and proofs,
please refer to the corresponding section of the paper.

### Expected runtime
With paper defaults (`GRID_RES = 200`, `MC_SAMPLES = 100`), the full notebook
runs in roughly **10 minutes** on a modern CPU laptop. The bottleneck is
`find_candidate_double_points` (one pairwise distance computation per model).
The configuration cell in §3.4 exposes the relevant knobs — set `GRID_RES = 80`
for a quick first pass.

### Pre-requisites
Run with cwd `surfaces_in_h4/`. The notebook expects the trained checkpoints
in `trained_models/paper/` (already shipped) and the Python environment built
from the repo's `requirements.txt`.


In [ ]:
import torch
import math
import matplotlib.pyplot as plt
import numpy as np

from src.full_models import HyperbolicMinimalSurfacePINN
from src.geometry import minimal_in_H4_PDE_flat_new
from src.samplers import MixSampler
from src.double_point_analysis import find_candidate_double_points, refine_double_points_newton
from src.plotting import plot_error, plot_knot, plot_mu_heatmap_log, montecarlo_error

%matplotlib inline

## §1 Introduction

The **asymptotic Plateau problem** in hyperbolic space asks: given a closed
submanifold $M^k \subset S^n$ of the boundary sphere of $\mathbb{H}^{n+1}$,
does there exist a $(k+1)$-dimensional properly immersed minimal submanifold
of $\mathbb{H}^{n+1}$ asymptotic to $M$? Anderson (1982) proved existence for
any closed $M$, but says nothing about the topology of the solution.

The paper focuses on the codimension-$2$ case: minimal **surfaces** in
$\mathbb{H}^4$ asymptotic to a **knot** $K \subset S^3$. Following Fine,
denote by $\mathcal{M}_{g,d}(K)$ the moduli space of connected, oriented,
properly immersed minimal surfaces in $\overline{\mathbb{H}}^4$ of genus $g$
and signed self-intersection number $d$, bounded by $K$ at infinity. Fine
proved that, for generic $K$, this is an oriented $0$-manifold; for $g = 0$
it is moreover compact, hence finite, and the signed count
$\#\mathcal{M}_{0,d}(K) \in \mathbb{Z}$ is a knot invariant.

### Fine's conjecture

> For a *generic* knot $K \subset S^3$, $\quad\displaystyle\#\mathcal{M}_{g,d}(K) \;=\; \big[z^{2g}\,a^{2(g+d)}\big]\,P(K),$
> where $P(K) \in \mathbb{Z}[a^{\pm 1}, z^{\pm 1}]$ is the HOMFLY polynomial
> of $K$.

This is the central testable claim. The present notebook produces
near-minimal **discs** ($g=0$) for 13 boundary knots and verifies that the
empirical signed count of double points matches the coefficient
$[a^{2d}]\,P(K)$ predicted by Fine's conjecture in every case considered in
the paper.


## §2 Minimal surfaces in hyperbolic 4-space

The paper develops the geometric setting in §2; we recall the points we use:

- Hyperbolic $(n+1)$-space in the **half-space model** is
  $\mathbb{H}^{n+1} = \{(X, Y_1, \dots, Y_n) : X > 0\}$ with metric
  $ds^2 = (dX^2 + |dY|^2)/X^2$. Its boundary at infinity is
  $\{X = 0\} \cong \mathbb{R}^n$ (which one-point-compactifies to $S^n$).
- An immersion $u : \Sigma \to N$ is **minimal** iff its **tension field**
  $\tau(u) = \mathrm{tr}_{u^* g}\,\nabla du$ vanishes. For $\Sigma$ a surface
  this is a second-order quasi-linear elliptic system.
- A **boundary defining function** (BDF) $\rho$ on $\bar M$ is a smooth
  nonnegative function vanishing exactly (and to first order) on
  $\partial \bar M$. A metric is **conformally compact** if it lives on the
  interior of $\bar M$ and extends smoothly after the conformal rescaling
  $\rho^2 g$. The half-space model is conformally compact with $\rho = X$;
  pulled back to the unit disc via the model's ansatz we use the
  **stereographic** BDF $\rho_{\mathrm{st}} = (1-r^2)/(1+r^2)$.
- A **p-immersion** $\Sigma \to \overline{\mathbb{H}}^{n+1}$ is a smooth
  boundary-respecting map (Melrose's "simple b-map") that is an immersion.
  The asymptotic Plateau problem asks for minimal p-immersions.
- For a generic p-immersion $u : D^2 \to \overline{\mathbb{H}}^4$ with
  $u_\partial$ an embedding, the image self-intersects only at finitely many
  transverse **double points** in the interior. Each is assigned a sign
  $\pm 1$ by comparing the orientation of $T_q\,\mathbb{H}^4$ against the
  four-vector union $du_{p_1}(T_{p_1}\Sigma) \cup du_{p_2}(T_{p_2}\Sigma)$.
  The **self-intersection number** $d \in \mathbb{Z}$ is the sum of these
  signs.

These notions are needed below to interpret the output of the Newton
refinement: it returns refined pre-image pairs together with the determinant
of the $4 \times 4$ Jacobian of $(p_1, p_2) \mapsto u(p_1) - u(p_2)$, whose
**sign** is the sign of the corresponding double point.


## §3 The Machine Learning approach: PINNs

### §3.1 Physics-Informed Neural Networks

A PINN parametrises candidate solutions of a PDE by a neural network
$\hat f_\theta$ (an MLP in our case) and trains $\theta$ to minimise the
squared PDE residual on a sample of interior **collocation points**. In a
standard formulation one would also add a boundary-fitting term. The
Universal Approximation Theorem (Hornik 1989) guarantees that any continuous
map on a compact domain is reachable in the space of MLPs in the $C^0$
topology.

The paper introduces a design choice that is crucial to convergence: rather
than training a generic MLP $\hat f_\theta : D^2 \to \overline{\mathbb{H}}^4$
with both interior and boundary loss, the MLP is embedded inside a composite
ansatz that **satisfies the boundary condition exactly, for every value of
the learnable parameters**. The loss reduces to an interior-only term.

### §3.2 The model

Every instance of the model is a map
$u_{\gamma,\rho,\mathrm{ext},k,\mathrm{NN}} : D^2 \to \overline{\mathbb{H}}^{n+1}$
of the form

$$
u_{\gamma,\rho,\mathrm{ext},k,\mathrm{NN}}(x,y) \;=\; \Big(\;\rho(x,y)\,\exp\big(\mathrm{NN}^X(x,y)\big),\;\; \mathrm{ext}(\gamma)(x,y) + \rho(x,y)^{k}\,\mathrm{NN}^Y(x,y)\;\Big).
$$

The five ingredients map directly to the code:

| Symbol | Role | Code |
|---|---|---|
| $\gamma : S^1 \to \mathbb{R}^3$ | Boundary knot parametrisation | `src.knots` — the `get_*_knot` factories, plus `generate_knot_perturbation_matrix` |
| $\rho$ | Boundary-defining function on $D^2$, here the stereographic BDF $\rho_\mathrm{st} = (1-r^2)/(1+r^2)$ | `src.full_models` (`bdf_type='stereographic'`); analytic derivatives in `src.geometry._stereographic_bdf_uJH` |
| $\mathrm{ext}$ | Smooth extension of $\gamma$ to $D^2$ — the stereo-biharmonic extension | `src.extensions.StereobiharmonicEvaluator`; analytic derivatives in `src.geometry._stereobiharmonic_ext_uJH_with_coeffs` |
| $k \in \{1, 2\}$ | Decay exponent | `decay_exponent` kwarg of `HyperbolicMinimalSurfacePINN` (default $2$) |
| $\mathrm{NN}$ | The interior MLP, $\mathbb{R}^2 \to \mathbb{R}^4$ | `src.interior_models.MLP` |

Because $\rho|_{\partial D^2} = 0$, the construction guarantees $X|_{\partial D^2} = 0$
and $Y|_{\partial D^2} = \gamma$ for every $\theta$ — the boundary condition is
satisfied by design. When all $\theta = 0$, the model reduces to the
**baseline** $(\rho_\mathrm{st}, \mathrm{ext}(\gamma))$. For the round unknot
$\gamma_\mathrm{unknot}(\theta) = (\cos\theta, \sin\theta, 0)$, this baseline
is the **totally geodesic** $\mathbb{H}^2 \hookrightarrow \mathbb{H}^4$ — the
exact minimal disc — so the unknot is the natural calibration case (see §4.1).

The architecture of $\mathrm{NN}$ used throughout is fixed:


In [ ]:
demo_kwargs = dict(
    knot_type='unknot',
    knot_kwargs={'R': 1.0},
    knot_perturbation_matrix=None,
    interior_model_kwargs=dict(in_dim=2, out_dim=4, hidden_dim=64, num_hidden_layers=4),
    bdf_type='stereographic',
    ext_type='stereobiharmonic',
    ext_kwargs={'N': 15, 'num_samples': 10000},
    decay_exponent=2,
)
demo_model = HyperbolicMinimalSurfacePINN(**demo_kwargs)
n_params = sum(p.numel() for p in demo_model.parameters())
print(f"Architecture: {demo_model.name}")
print(f"  hidden width W       = {demo_kwargs['interior_model_kwargs']['hidden_dim']}")
print(f"  hidden depth L       = {demo_kwargs['interior_model_kwargs']['num_hidden_layers']}")
print(f"  trainable parameters = {n_params}  (paper §3.2.1 reports 12932)")

### §3.2.1 The hyperparameter choice (paper Table 1)

| Symbol | Description | Default | Phase |
|---|---|---|---|
| $W$ | Hidden-layer width | $64$ | all |
| $L$ | Hidden depth | $4$ | all |
| $\sigma$ | Activation | $\tanh$ | all |
| $N_\mathrm{data}$ | Collocation pool size | $2^{14}$ | 1, 2 |
| $B$ | Mini-batch size | $2^{10}$ | 1 |
| $T_2$ | Adam epochs | $10\,000$ | 1 |
| $\eta_0$ | Initial learning rate | $10^{-3}$ | 1 |
| $\eta_\mathrm{min}$ | Minimum learning rate | $10^{-5}$ | 1 |
| $N_\mathrm{LBFGS}$ | L-BFGS collocation size | $2^{14}$ | 2 |
| $T_\mathrm{LBFGS}$ | L-BFGS max iterations | $10\,000$ | 2 |
| $m$ | L-BFGS history size | $100$ | 2 |
| $\delta_g$ | Gradient-norm tolerance | $10^{-12}$ | 2 |
| $\delta_\theta$ | Parameter-change tolerance | $10^{-14}$ | 2 |

All experiments run in `float64` and on CPU only.


### §3.3 The training regime

Training proceeds in two phases (cf. `src.training`):

1. **Adam with cosine-annealed learning rate** (`train_PINN_Adam`): a pool of
   $N_\mathrm{data} = 2^{14}$ collocation points sampled by $r = U^{1/2}$,
   $\varphi = 2\pi V$ is drawn once; mini-batches of size $B = 2^{10}$ are
   passed through the compiled PDE-residual evaluator for $T_2 = 10\,000$
   epochs. The snapshot with the lowest epoch-averaged loss is retained.
2. **Full-batch L-BFGS with strong-Wolfe line search** (`refine_PINN_lbfgs`):
   a fresh static sample of $N_\mathrm{LBFGS} = 2^{14}$ points is drawn;
   L-BFGS runs for at most $T_\mathrm{LBFGS} = 10\,000$ steps, with strict
   gradient-norm and parameter-change tolerances.

Re-running this from scratch is **not** required to reproduce the paper —
the trained checkpoints are shipped in `trained_models/paper/`. The notebook
[`train_and_save.ipynb`](train_and_save.ipynb) demonstrates the recipe for
building, training, and saving a model for a new knot. A full Adam + L-BFGS
cycle takes roughly an hour on a modern CPU laptop.


### §3.4 Double-point analysis

Given a near-minimal disc $u : D^2 \to \overline{\mathbb{H}}^4$, the
algorithm to find self-intersections runs in two stages.

1. **Candidate generation.** Define the **self-proximity map**
   $\mu_\varepsilon : D^2 \to [0, \infty)$ at threshold $\varepsilon > 0$:
   $$
   \mu_\varepsilon(p) \;=\; \inf_{\substack{p' \in D^2 \\ \|p - p'\| > \varepsilon}} \|u(p) - u(p')\|.
   $$
   On a regular grid $\{p_i\} \subset D^2$, retain the pairs $(p_i, p_j)$ with
   $\|p_i - p_j\| > \varepsilon$ and $\|u(p_i) - u(p_j)\| < \tau$.
   *(`find_candidate_double_points`)*
2. **Newton refinement.** Solve $F_u(p_1, p_2) := u(p_1) - u(p_2) = 0$,
   $F_u : \mathbb{R}^2 \times \mathbb{R}^2 \to \mathbb{R}^4$, by Newton's
   method starting from each candidate. The **sign of** $\det J_{F_u}$ at
   each converged root is the contribution of that double point to the
   self-intersection number $d$.
   *(`refine_double_points_newton`)*

The cell below exposes the analysis settings. Reduce `GRID_RES` and/or
`MC_SAMPLES` for faster iteration.


In [ ]:
# === Configuration ===
GRID_RES        = 200          # candidate-finder grid resolution (paper: 200)
EPS             = 0.2          # domain threshold for the candidate finder
TAU             = 0.03         # codomain threshold for the candidate finder
HEATMAP_EPS     = 0.3          # epsilon for the displayed mu_eps heatmap
HEATMAP_RES     = 200          # resolution for the mu_eps heatmap

NEWTON_TOL      = 1e-14
NEWTON_MAX_ITER = 200
DEDUP_TOL       = 1e-11

N_POINTS        = 2 ** 14      # sample size for tension-field metrics
MC_SAMPLES      = 1000         # Monte Carlo replicates (paper uses 1000)

MODELS_DIR      = "trained_models/paper"


def analyze(file_name, *, label, homfly_term, predicted_d):
    '''Load a trained model, evaluate it, find and refine double points.

    Returns a dict bundling the model, error metrics, refined pre-image
    pairs (with Jacobian signs), and the resulting signed count d.
    '''
    path = f"{MODELS_DIR}/{file_name}"
    model = HyperbolicMinimalSurfacePINN.load(path)
    dtype = model.kwargs['dtype']

    # Tension-field metrics (MSE and pointwise max).
    sampler = MixSampler(mix=1, bias=0.5, dtype=dtype)
    xy = sampler(N_POINTS)
    t = minimal_in_H4_PDE_flat_new(model)(xy)
    mse     = (t ** 2).sum(dim=-1).mean().item()
    max_tau = torch.max((t ** 2).sum(dim=-1) ** 0.5).item()

    # Candidate double points.
    candidates = find_candidate_double_points(
        model, grid_resolution=GRID_RES, epsilon=EPS, tau=TAU, dtype=dtype,
    )

    # Newton refinement.
    if len(candidates) > 0:
        refined, jacs, dists = refine_double_points_newton(
            model, candidates,
            tol=NEWTON_TOL, max_iter=NEWTON_MAX_ITER, dedup_tol=DEDUP_TOL,
            dtype=dtype,
        )
    else:
        refined, jacs, dists = [], [], []

    n_pos = sum(1 for j in jacs if j > 0)
    n_neg = sum(1 for j in jacs if j < 0)
    d     = n_pos - n_neg

    return {
        'label':              label,
        'file':               file_name,
        'homfly_term':        homfly_term,
        'predicted_d':        predicted_d,
        'model':              model,
        'mse':                mse,
        'max_tau':            max_tau,
        'candidates':         candidates,
        'refined_pairs':      refined,
        'jacobians':          jacs,
        'final_distances':    dists,
        'n_positive':         n_pos,
        'n_negative':         n_neg,
        'self_intersection':  d,
        'matches_prediction': d == predicted_d,
    }


def summary_line(res):
    '''Compact textual summary of an `analyze(...)` result.'''
    label = res['label']
    print(f"[{label}]  MSE = {res['mse']:.2e}, max|tau| = {res['max_tau']:.2e}")
    print(f"          double points: {res['n_positive']} positive, {res['n_negative']} negative")
    print(f"          self-intersection number d = {res['self_intersection']}")
    print(f"          Fine's prediction (term {res['homfly_term']}): d = {res['predicted_d']}")
    print(f"          match: {'YES' if res['matches_prediction'] else 'NO'}")


def show_mu_heatmap(res, *, with_candidates=False, **overrides):
    '''Plot the log mu_eps heatmap with refined double points overlaid.'''
    kw = dict(
        u_callable           = res['model'],
        epsilon              = HEATMAP_EPS,
        grid_resolution      = HEATMAP_RES,
        cmap                 = 'viridis_r',
        figsize              = (6, 5),
        boundary_linewidth   = 2.6,
        refined_pairs        = res['refined_pairs'],
        jacobians            = res['jacobians'],
        refined_pos_color    = '#ffc000',
        refined_neg_color    = '#fe0c0c',
        legend_label_refined_pos = r'Positive double point',
        legend_label_refined_neg = r'Negative double point',
        legend_label_candidates  = r'Candidate pair',
        legend_frameon       = True,
        show_colorbar        = True,
        colorbar_label       = None,
        title                = None,
    )
    if with_candidates:
        kw['candidates']      = res['candidates']
        kw['candidate_color'] = "#E85D04"
        kw['alpha']           = 0.7
    kw.update(overrides)
    plot_mu_heatmap_log(**kw)

## §4 Results

For each of the 13 knots considered in the paper, we (i) load the trained
checkpoint from `trained_models/paper/`, (ii) record the tension-field MSE
and pointwise max, (iii) find the double points and refine them by Newton's
method, (iv) plot the log self-proximity heatmap with refined pre-images
overlaid, (v) compare the resulting signed count $d$ with the prediction
$[a^{2d}]\,P(K)$ from Fine's conjecture. The §4 summary cell at the end
collects all 13 rows into a table that mirrors paper Table 2.


### §4.1 Unknot — calibration

We use the round unknot $\gamma_\mathrm{unknot}(\theta) = (\cos\theta, \sin\theta, 0)$,
deformed by a small Fourier perturbation. The corresponding totally geodesic disc

$$
u_{\mathbb{H}^2}(\theta, r) = \Big(\tfrac{1 - r^2}{1 + r^2},\; \tfrac{2 r \cos\theta}{1 + r^2},\; \tfrac{2 r \sin\theta}{1 + r^2},\; 0\Big)
$$

is the **exact** minimal disc and lies *within* the model class — so the
unknot run is a calibration of the framework.

HOMFLY polynomial: $P(\mathrm{unknot}) = 1$, predicting $d = 0$ (an embedded
disc).


In [ ]:
unknot_file = "KNOT_unknot_KNOT_PAR_R1_PERTURBED.pt"
unknot_path = f"{MODELS_DIR}/{unknot_file}"

# Load the trained model and instantiate an untrained sibling with the same
# config (= fresh random weights) for the "before" plot.
unknot_trained   = HyperbolicMinimalSurfacePINN.load(unknot_path)
unknot_untrained = HyperbolicMinimalSurfacePINN(**unknot_trained.kwargs)

print("|tau(u_unknot)|^2 -- BEFORE training:")
plot_error(
    minimal_in_H4_PDE_flat_new(unknot_untrained),
    dtype=unknot_untrained.kwargs['dtype'],
    grid_size=200, vmin=0, vmax=1, figsize=(6, 5),
    boundary_linewidth=2.6, colorbar_label='', title=None,
)

print("\\n|tau(u_unknot)|^2 -- AFTER training:")
plot_error(
    minimal_in_H4_PDE_flat_new(unknot_trained),
    dtype=unknot_trained.kwargs['dtype'],
    grid_size=200, vmin=0, vmax=1, figsize=(6, 5),
    boundary_linewidth=2.6, colorbar_label='', title=None,
)
print("Note: the residual is exactly zero on the boundary even BEFORE training,")
print("by the ansatz design (paper §3.2).")

In [ ]:
print(f"Monte Carlo simulation of L(u_unknot; .) -- {MC_SAMPLES} samples of size {N_POINTS}:")
montecarlo_error(
    minimal_in_H4_PDE_flat_new(unknot_trained),
    num_samples=MC_SAMPLES, size_samples=N_POINTS,
    mix=1, bias=0.5,
    figsize=(6, 3), bins=40, title=None, xlabel='Loss',
)

In [ ]:
res_unknot = analyze(unknot_file, label='unknot', homfly_term='1', predicted_d=0)
summary_line(res_unknot)

### §4.2 Torus knots

The $(p, q)$ torus knot has the standard parametrisation

$$
\gamma_{p,q}(\theta) \;=\; \Big((R + r\cos(q\theta))\cos(p\theta),\;(R + r\cos(q\theta))\sin(p\theta),\;r\sin(q\theta)\Big).
$$

The paper trains five torus solutions ($3_1$, $5_1$, $8_{19}$ unperturbed and
perturbed, $10_{124}$) and notes a general pattern: for $(p, q)$ torus knots
with $p, q > 0$, the smallest-degree HOMFLY monomial is

$$
\tfrac{1}{p+q}\,\binom{p+q}{p}\,a^{(p-1)(q-1)},
$$

and the algorithm consistently finds a solution with $\tfrac{(p-1)(q-1)}{2}$
positive double points — coinciding with the smooth slice genus, i.e. the
*minimum possible* number of double points for an immersed disc bounding the
$(p, q)$ torus knot.


#### $3_1$ — right-handed trefoil

The $(3, 2)$ torus knot. HOMFLY:
$P(3_1) = 2 a^2 - a^4 + a^2 z^2$.
The term $2 a^2$ predicts $d = 1$. *(Paper §4.2, symbol $u_{3,2}$.)*


In [ ]:
res_3_1 = analyze(
    "KNOT_torus_KNOT_PAR_R1_p3_q2_r0.5_PERTURBED.pt",
    label='3_1', homfly_term='2a^2', predicted_d=1,
)
summary_line(res_3_1)
show_mu_heatmap(res_3_1, with_candidates=True)

#### $8_{19}$ and the role of perturbation

The $(4, 3)$ torus knot, $8_{19}$. HOMFLY:
$P(8_{19}) = (5 a^6 - 5 a^8 + a^{10}) + (10 a^6 - 5 a^8) z^2 + (6 a^6 - a^8) z^4 + a^6 z^6$.
The term $5 a^6$ predicts $d = 3$.

The unperturbed parametrisation $\gamma_{4,3}$ is non-generically symmetric;
the trained solution $u_{4,3}$ exhibits a **triple point** (a single
self-intersection with three pre-images). Adding a small Fourier perturbation
yields an isotopic boundary knot whose trained solution $\tilde u_{4,3}$
instead has **three transverse double points**, as expected for the generic
desingularisation of a triple point (paper §2.3 Fig 4).


In [ ]:
# Unperturbed: triple point -- the algorithm will report multiple
# candidate pairs clustered around the single non-generic singularity.
res_8_19_unperturbed = analyze(
    "KNOT_torus_KNOT_PAR_R1_p4_q3_r0.5.pt",
    label='unperturbed 8_19', homfly_term='5a^6', predicted_d=3,
)
summary_line(res_8_19_unperturbed)
print("(Triple-point case: candidate pairs cluster around a single non-generic intersection;")
print(" Newton may converge to closely related roots near the same triple point.)")
show_mu_heatmap(res_8_19_unperturbed)

In [ ]:
# Perturbed: three clean transverse double points.
res_8_19 = analyze(
    "KNOT_torus_KNOT_PAR_R1_p4_q3_r0.5_PERTURBED.pt",
    label='8_19', homfly_term='5a^6', predicted_d=3,
)
summary_line(res_8_19)
show_mu_heatmap(res_8_19)

#### $5_1$

The $(5, 2)$ torus knot. HOMFLY (low order in $z$):
$P(5_1) = 3 a^4 - 2 a^6 + O(z^2)$.
The term $3 a^4$ predicts $d = 2$.


In [ ]:
res_5_1 = analyze(
    "KNOT_torus_KNOT_PAR_R1_p5_q2_r0.5_PERTURBED.pt",
    label='5_1', homfly_term='3a^4', predicted_d=2,
)
summary_line(res_5_1)
show_mu_heatmap(res_5_1)

#### $10_{124}$

The $(5, 3)$ torus knot, the most complex example in the paper (crossing
number $10$). HOMFLY (low order in $z$):
$P(10_{124}) = 7 a^8 - 8 a^{10} + 2 a^{12} + O(z^2)$.
The term $7 a^8$ predicts $d = 4$.


In [ ]:
res_10_124 = analyze(
    "KNOT_torus_KNOT_PAR_R1_p5_q3_r0.5_PERTURBED.pt",
    label='10_124', homfly_term='7a^8', predicted_d=4,
)
summary_line(res_10_124)
show_mu_heatmap(res_10_124)

### §4.3 Other knots

Four non-torus examples, each illustrating a distinct feature of Fine's
conjecture: the figure-eight (achiral, illustrating the $a^{-2} \leftrightarrow a^2$
duality), the three-twist (chiral example with a non-minimal-$d$ solution),
the Stevedore (mixed-sign double points; ribbon knot), and the square knot
$3_1 \# 3_1^*$ (a non-prime example with cancelling double points).


#### $4_1$ — figure-eight

The figure-eight knot is the simplest **achiral** knot ($4_1 \sim 4_1^*$).
The boundary parametrisation used is the modulated-torus form

$$
\gamma_\mathrm{figure8}(\theta) \;=\; \Big((1 + \tfrac12 \cos 2\theta)\cos 3\theta,\; (1 + \tfrac12 \cos 2\theta)\sin 3\theta,\; \tfrac12 \sin 4\theta\Big),
$$

implemented as `get_figure8_torus`. HOMFLY:
$P(4_1) = a^{-2} - 1 + a^2 - z^2$.
The term $a^2$ predicts $d = 1$; replacing $\gamma$ by $-\gamma$ (an
equivalent parametrisation of $4_1^*$) gives a solution with $d = -1$,
predicted by the term $a^{-2}$.


In [ ]:
res_4_1 = analyze(
    "KNOT_figure8_torus_KNOT_PAR_A0.45_R1_mirrorFalse_PERTURBED.pt",
    label='4_1', homfly_term='a^2', predicted_d=1,
)
summary_line(res_4_1)
show_mu_heatmap(res_4_1)

res_4_1_star = analyze(
    "KNOT_figure8_torus_KNOT_PAR_A0.45_R1_mirrorTrue_PERTURBED.pt",
    label='4_1* (~ 4_1)', homfly_term='a^{-2}', predicted_d=-1,
)
summary_line(res_4_1_star)
show_mu_heatmap(res_4_1_star)

#### $5_2$ — three-twist

Lissajous parametrisation
$\gamma_\mathrm{3twist}(\theta) = (-\cos(3\theta + 0.7),\; -\cos(2\theta + 0.2),\; -\cos 7\theta)$,
implemented as `get_three_twist_knot`. HOMFLY:
$P(5_2) = (a^2 + a^4 - a^6) + (a^2 + a^4) z^2$.
The term $-a^6$ predicts $d = 3$. The mirror gives $d = -3$.

The slice genus of $5_2$ is $1$, so a solution with $d = 1$ would saturate
that lower bound; the paper instead finds a $d = 3$ solution corresponding to
the $-a^6$ term.


In [ ]:
res_5_2 = analyze(
    "KNOT_three_twist_KNOT_PAR_R0.6_mirrorFalse_PERTURBED.pt",
    label='5_2', homfly_term='-a^6', predicted_d=3,
)
summary_line(res_5_2)
show_mu_heatmap(res_5_2)

res_5_2_star = analyze(
    "KNOT_three_twist_KNOT_PAR_R0.6_mirrorTrue_PERTURBED.pt",
    label='5_2*', homfly_term='-a^{-6}', predicted_d=-3,
)
summary_line(res_5_2_star)
show_mu_heatmap(res_5_2_star)

#### $6_1$ — Stevedore

Lissajous parametrisation
$\gamma_\mathrm{Stevedore}(\theta) = (-\cos(3\theta + 1.5),\; -\cos(2\theta + 0.2),\; -\cos 5\theta)$,
implemented as `get_stevedore_knot`. HOMFLY:
$P(6_1) = (a^{-2} - a^2 + a^4) + (-1 - a^2) z^2$.
The term $-a^2$ predicts $d = 1$, and the algorithm should find a mixed
configuration of **two positive and one negative** double points.

The paper's remark (L2290) is worth recalling: although $6_1$ *is* smoothly
slice (so an embedded disc bounding it exists), Fine's conjecture is
consistent with this — the constant term of $P(6_1)$ is $0$, which forecasts
an *even* number of $d = 0$ solutions whose signs cancel, not the
non-existence of an embedded one.


In [ ]:
res_6_1 = analyze(
    "KNOT_stevedore_KNOT_PAR_R1.6_mirrorFalse_PERTURBED.pt",
    label='6_1', homfly_term='-a^2', predicted_d=1,
)
summary_line(res_6_1)
show_mu_heatmap(res_6_1)

res_6_1_star = analyze(
    "KNOT_stevedore_KNOT_PAR_R0.95_mirrorTrue_PERTURBED.pt",
    label='6_1*', homfly_term='-a^{-2}', predicted_d=-1,
)
summary_line(res_6_1_star)
show_mu_heatmap(res_6_1_star)

#### $3_1 \# 3_1^*$ — square knot

The only non-prime example: the connected sum of a right-handed trefoil
$3_1$ and a left-handed trefoil $3_1^*$. Lissajous parametrisation
$\gamma_\mathrm{square}(\theta) = (\cos(3\theta + 0.7),\; \cos(5\theta + 1),\; \cos 7\theta)$,
implemented as `get_square_knot`. HOMFLY:

$$
P(3_1 \# 3_1^*) = P(3_1)\,P(3_1^*) = (-2 a^{-2} + 5 - 2 a^2) + (-a^{-2} + 4 - a^2) z^2 + z^4.
$$

The constant term $5$ predicts $d = 0$; the algorithm should find **two
positive and two negative** double points whose signs cancel.


In [ ]:
res_square = analyze(
    "KNOT_square_KNOT_PAR_R0.6_PERTURBED.pt",
    label='3_1 # 3_1*', homfly_term='5', predicted_d=0,
)
summary_line(res_square)
show_mu_heatmap(res_square)

### §4 Summary

The following table reproduces paper Table 2 (L2321–L2358) from the computed
results above. The "Computed $d$" column should match the "Predicted $d$"
column in every row except the unperturbed $8_{19}$, where the non-generic
triple point precludes a clean count (the paper marks this entry "triple
point" rather than a signed integer).


In [ ]:
# Collect only the per-knot result variables that have actually been defined
# in this kernel. Run whichever §4 subsections you care about; this cell
# picks up the corresponding res_* and silently skips the rest, preserving
# the paper's row ordering.
_canonical_order = [
    'res_unknot',
    'res_3_1',
    'res_5_1',
    'res_8_19',
    'res_8_19_unperturbed',
    'res_10_124',
    'res_4_1',
    'res_4_1_star',
    'res_5_2',
    'res_5_2_star',
    'res_6_1',
    'res_6_1_star',
    'res_square',
]

_ns = globals()
all_results = [_ns[name] for name in _canonical_order if name in _ns]

if not all_results:
    print("No per-knot results defined yet -- run at least one §4 subsection first.")
else:
    header = f"{'Knot':18s}  {'HOMFLY term':12s}  {'Pred d':>6s}  {'Found d':>7s}  {'(#+, #-)':>10s}  {'MSE':>10s}  Match"
    print(header)
    print('-' * len(header))
    for r in all_results:
        match = 'YES' if r['matches_prediction'] else ('--' if 'unperturbed' in r['label'] else 'NO')
        pm = f"({r['n_positive']:>2d}, {r['n_negative']:>2d})"
        print(f"{r['label']:18s}  {r['homfly_term']:12s}  {r['predicted_d']:>6d}  {r['self_intersection']:>7d}  {pm:>10s}  {r['mse']:>10.2e}  {match}")

    n_run = len(all_results)
    n_total = len(_canonical_order)
    print()
    if n_run == n_total:
        print(f"All {n_total} rows present.")
    else:
        skipped = [n for n in _canonical_order if n not in _ns]
        print(f"({n_run} of {n_total} rows present; not yet run: {', '.join(s.removeprefix('res_') for s in skipped)})")